# SpaceX Falcon 9 first-stage landing analysis

Independent reproduction using IBM's course datasets. `dataset_part_2.csv` has 90 model records, while the separate SQL data has 101 records. Landing outcome differs from mission outcome.

In [1]:
import pandas as pd
import sqlite3
import json
from pathlib import Path

df = pd.read_csv('dataset_part_2.csv')
print(f'{len(df)} launches, {df.Date.min()} to {df.Date.max()}')
df[['Date', 'LaunchSite', 'Orbit', 'PayloadMass', 'Outcome', 'Class']].head()

90 launches, 2010-06-04 to 2020-11-05


,Date,LaunchSite,Orbit,PayloadMass,Outcome,Class
0,2010-06-04,CCAFS SLC 40,LEO,6104.959412,None None,0
1,2012-05-22,CCAFS SLC 40,LEO,525.000000,None None,0
2,2013-03-01,CCAFS SLC 40,ISS,677.000000,None None,0
3,2013-09-29,VAFB SLC 4E,PO,500.000000,False Ocean,0
4,2013-12-03,CCAFS SLC 40,GTO,3170.000000,None None,0


## Data collection demonstrations
`data_collection.py` attempts the SpaceX v4 API, and falls back to the course's static API-response snapshot if the live endpoint is unavailable. It also scrapes the fixed 2021 Wikipedia revision. These two samples are not used as direct inputs to the 90-row modeling data.

In [2]:
api = pd.read_csv('api_launch_sample.csv')
web = pd.read_csv('wiki_launch_sample.csv')
print(f'API-response sample: {len(api)} rows; Wikipedia sample: {len(web)} rows')
web.head(3)

API-response sample: 107 rows; Wikipedia sample: 121 rows


,flight,date_time_utc,version_booster,launch_site,payload,payload_mass,orbit,customer,launch_outcome,booster_landing
0,1,"4 June 2010, 18:45",F9 v1.0 B0003.1,"CCAFS , SLC-40",Dragon Spacecraft Qualification Unit,NaN,LEO,SpaceX,Success,Failure (parachute)
1,2,"8 December 2010, 15:43",F9 v1.0 B0004.1,"CCAFS , SLC-40",Dragon demo flight C1 (Dragon C101),NaN,LEO ( ISS ),NASA ( COTS ) NRO,Success,Failure (parachute)
2,3,"22 May 2012, 07:44",F9 v1.0 B0005.1,"CCAFS , SLC-40",Dragon demo flight C2+ (Dragon C102),"525 kg (1,157 lb)",LEO ( ISS ),NASA ( COTS ),Success,No attempt


## Exploratory analysis
Class 1 includes successful ground, drone-ship or controlled ocean outcomes. Class 0 also includes records with no landing attempt.

In [3]:
print('Overall counts:', df.Class.value_counts().to_dict())
df.groupby('LaunchSite').Class.agg(launches='count', successes='sum', success_rate='mean')

Overall counts: {1: 60, 0: 30}


,launches,successes,success_rate
LaunchSite,,,
CCAFS SLC 40,55,33,0.600000
KSC LC 39A,22,17,0.772727
VAFB SLC 4E,13,10,0.769231


In [4]:
df.groupby('Orbit').Class.agg(launches='count', successes='sum', success_rate='mean').sort_values('launches', ascending=False)

,launches,successes,success_rate
Orbit,,,
GTO,27,14,0.518519
ISS,21,13,0.619048
VLEO,14,12,0.857143
PO,9,6,0.666667
LEO,7,5,0.714286
SSO,5,5,1.000000
MEO,3,2,0.666667
GEO,1,1,1.000000
ES-L1,1,1,1.000000


## SQL analysis
The SQL queries use a separate course CSV. Site labels and row counts are those in that source.

In [5]:
raw_sql = pd.read_csv('spacex_sql.csv', encoding='latin-1')
with sqlite3.connect(':memory:') as connection:
    raw_sql.to_sql('launches', connection, index=False)
    sites = pd.read_sql_query('SELECT Launch_Site, COUNT(*) AS launches FROM launches GROUP BY Launch_Site ORDER BY launches DESC', connection)
sites

,Launch_Site,launches
0,CCAFS SLC-40,34
1,CCAFS LC-40,26
2,KSC LC-39A,25
3,VAFB SLC-4E,16


## Classification
The script fits preprocessing within each training fold and compares models using four-fold cross-validation. It then evaluates the fixed models on an 18-row holdout. The provided CSV already includes some payload imputations.

In [6]:
from analyze import main
main()

DATA (90, 18) 2010-06-04 2020-11-05 CLASS {1: 60, 0: 30}
SITE               count  sum      mean
LaunchSite                        
CCAFS SLC 40     55   33  0.600000
KSC LC 39A       22   17  0.772727
VAFB SLC 4E      13   10  0.769231
ORBIT        count  sum      mean
Orbit                      
ES-L1      1    1  1.000000
GEO        1    1  1.000000
GTO       27   14  0.518519
HEO        1    1  1.000000
ISS       21   13  0.619048
LEO        7    5  0.714286
MEO        3    2  0.666667
PO         9    6  0.666667
SO         1    0  0.000000
SSO        5    5  1.000000
VLEO      14   12  0.857143
YEAR       count  sum      mean
Year                      
2010      1    0  0.000000
2012      1    0  0.000000
2013      3    0  0.000000
2014      6    2  0.333333
2015      6    2  0.333333
2016      8    5  0.625000
2017     18   15  0.833333
2018     18   11  0.611111
2019     10    9  0.900000
2020     19   16  0.842105
REUSE         count  sum      mean
Reused                      


MODELS {
  "Majority baseline": {
    "cv_accuracy_mean": 0.6666666666666666,
    "cv_accuracy_std": 0.0,
    "test_accuracy": 0.6666666666666666,
    "confusion_matrix": [
      [
        0,
        6
      ],
      [
        0,
        12
      ]
    ],
    "test_report": {
      "0": {
        "precision": 0.0,
        "recall": 0.0,
        "f1-score": 0.0,
        "support": 6.0
      },
      "1": {
        "precision": 0.6666666666666666,
        "recall": 1.0,
        "f1-score": 0.8,
        "support": 12.0
      },
      "accuracy": 0.6666666666666666,
      "macro avg": {
        "precision": 0.3333333333333333,
        "recall": 0.5,
        "f1-score": 0.4,
        "support": 18.0
      },
      "weighted avg": {
        "precision": 0.4444444444444444,
        "recall": 0.6666666666666666,
        "f1-score": 0.5333333333333334,
        "support": 18.0
      }
    }
  },
  "Logistic regression": {
    "cv_accuracy_mean": 0.6666666666666667,
    "cv_accuracy_std": 0.087841

In [7]:
results = json.loads(Path('model_results.json').read_text(encoding='utf-8'))
pd.DataFrame({name: {'CV accuracy': value['cv_accuracy_mean'], 'Holdout accuracy': value['test_accuracy']} for name, value in results.items()}).T

,CV accuracy,Holdout accuracy
Majority baseline,0.666667,0.666667
Logistic regression,0.666667,0.888889
SVM,0.722222,0.777778
Decision tree,0.750000,0.777778
KNN,0.666667,0.777778


## Interactive visual analytics
Open `launch_sites_map.html` for the Folium map. Run `python visual_analytics.py` from the project folder to start the Dash app. See the README for setup and limitations.